# 04 · Dynamics stability (theme D)

The preceding notebooks establish that virtual sites are *exported* correctly.
This notebook confirms the exported systems are also **dynamically stable and
physically sane** when actually integrated — the specific combination of tests
(virtual sites in both solvent and solute, production charges, and short MD to
confirm parameters behave as expected) that OpenFE asked OpenFF to provide
before building virtual-site free-energy support.

Initial coordinates are generated with **Packmol** (via
`openff.interchange.components._packmol`), the same solvation route users take.
We run short OpenMM simulations of Interchange-exported systems and check:
finite/stable energies, an equilibrated temperature, a physically reasonable
density, satisfied rigid-water constraints, and virtual sites that track their
parents (no drift/NaN). A final check confirms the Interchange- and
`openmmforcefields`-built water boxes agree on the potential energy.

In [1]:
import warnings

import numpy as np
import openmm
import openmm.unit as u
from openff.toolkit import ForceField as OFF
from openff.toolkit import Molecule
from openff.units import unit as offunit
from openff.interchange.components._packmol import pack_box, UNIT_CUBE
from openmm import LangevinMiddleIntegrator, MonteCarloBarostat
from openmm.app import Element, Modeller, PME
from openmm.app import ForceField as OMMFF
from openmm.app import Topology as OMMTopology

import _helpers as H

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
RESULTS = []
KB = u.MOLAR_GAS_CONSTANT_R


def pack(molecules, counts, density_kgm3=1000.0):
    """Packmol-generate a cubic box. Returns (off_topology, positions_nm_Quantity, box_nm)."""
    top = pack_box(molecules=molecules, number_of_copies=counts,
                   target_density=density_kgm3 * offunit.kilogram / offunit.meter ** 3,
                   box_shape=UNIT_CUBE)
    pos = np.asarray(top.get_positions().m_as(offunit.nanometer))
    box = np.asarray(top.box_vectors.m_as(offunit.nanometer))
    return top, pos * u.nanometer, box


def ligand_water_ff(small_molecule_ff="openff-2.3.0.offxml"):
    """Small-molecule FF (NAGL charges) + OPC water + a BondCharge sigma-hole on C-Cl."""
    ff = OFF(small_molecule_ff, "opc.offxml")
    ff.get_parameter_handler("VirtualSites").add_parameter(dict(
        smirks="[#6:1]-[#17:2]", type="BondCharge", name="EP", match="all_permutations",
        distance=-0.35 * offunit.angstrom,
        charge_increment1=0.0 * offunit.elementary_charge,
        charge_increment2=0.05 * offunit.elementary_charge,
        sigma=3.0 * offunit.angstrom, epsilon=0.02 * offunit.kilocalorie / offunit.mole))
    return ff


def simulate(system, positions, box, *, steps_nvt=1000, steps_npt=1000, dt_fs=1.0,
             temp=300, barostat=True, platform="CPU"):
    """Minimize, NVT, then optional NPT. Return a dict of stability diagnostics."""
    system = openmm.XmlSerializer.deserialize(openmm.XmlSerializer.serialize(system))
    # Clamp the nonbonded cutoff to fit the (small) test box; fine for a stability
    # check. Leave headroom for the barostat shrinking the box.
    box_min = float(np.min(np.diag(box)))
    cutoff = max(0.4, box_min * (0.42 if barostat else 0.48))
    for f in system.getForces():
        if isinstance(f, openmm.NonbondedForce):
            f.setUseSwitchingFunction(False)
            f.setCutoffDistance(cutoff * u.nanometer)
    if barostat:
        system.addForce(MonteCarloBarostat(1.0 * u.bar, temp * u.kelvin, 25))
    integ = LangevinMiddleIntegrator(temp * u.kelvin, 1.0 / u.picosecond, dt_fs * u.femtosecond)
    ctx = openmm.Context(system, integ, openmm.Platform.getPlatformByName(platform))
    ctx.setPeriodicBoxVectors(*(box * u.nanometer))
    ctx.setPositions(H._pad_positions(positions, system.getNumParticles()))
    ctx.computeVirtualSites()
    openmm.LocalEnergyMinimizer.minimize(ctx, maxIterations=200)
    e0 = ctx.getState(getEnergy=True).getPotentialEnergy().value_in_unit(H.ENERGY_UNIT)

    integ.step(steps_nvt)
    e_nvt = ctx.getState(getEnergy=True).getPotentialEnergy().value_in_unit(H.ENERGY_UNIT)

    energies, temps = [], []
    ndof = 3 * sum(system.getParticleMass(i).value_in_unit(u.dalton) > 0
                   for i in range(system.getNumParticles())) - system.getNumConstraints() - 3
    blocks = 10
    for _ in range(blocks):
        integ.step((steps_npt // blocks) if barostat else (steps_nvt // blocks))
        s = ctx.getState(getEnergy=True)
        energies.append(s.getPotentialEnergy().value_in_unit(H.ENERGY_UNIT))
        temps.append((2 * s.getKineticEnergy() / (ndof * KB)).value_in_unit(u.kelvin))

    final = ctx.getState(getEnergy=True, getPositions=True, enforcePeriodicBox=True)
    box_final = final.getPeriodicBoxVectors(asNumpy=True).value_in_unit(u.nanometer)
    vol = np.linalg.det(box_final)
    total_mass = sum(system.getParticleMass(i).value_in_unit(u.dalton)
                     for i in range(system.getNumParticles()))
    density = (total_mass / 6.02214076e23) / (vol * 1e-21)   # g/cm^3
    pos = final.getPositions(asNumpy=True).value_in_unit(u.nanometer)
    finite = bool(np.all(np.isfinite(pos)) and np.isfinite(energies[-1]))
    del ctx, integ
    return {"e_min": e0, "e_nvt": e_nvt, "e_series": energies, "T_mean": float(np.mean(temps)),
            "T_std": float(np.std(temps)), "density": float(density), "volume": float(vol),
            "finite": finite}

## D1 · Pure virtual-site water boxes (Packmol)

OPC (4-site) and TIP5P (5-site) boxes are packed with Packmol, built by
Interchange, minimized, and run NVT→NPT. We require finite energies, a
temperature near the 300 K setpoint, and a physical density.

In [2]:
def run_water(model, offxml, n_waters=150):
    top, pos, box = pack([Molecule.from_smiles("O")], [n_waters], density_kgm3=1000.0)
    system = OFF(offxml).create_interchange(top).to_openmm_system()
    diag = simulate(system, pos, box, steps_nvt=800, steps_npt=1500)
    print(f"\n[{model}] {n_waters} waters, {system.getNumParticles()} particles")
    print(f"  E_min={diag['e_min']:.1f}  E_nvt={diag['e_nvt']:.1f} kcal/mol  finite={diag['finite']}")
    print(f"  T={diag['T_mean']:.1f}+/-{diag['T_std']:.1f} K  density={diag['density']:.3f} g/cm^3")
    assert diag["finite"], f"{model}: non-finite dynamics"
    assert 200 < diag["T_mean"] < 400, f"{model}: temperature {diag['T_mean']} off setpoint"
    assert 0.6 < diag["density"] < 1.4, f"{model}: unphysical density {diag['density']}"
    RESULTS.append({"notebook": "04", "system": f"{model} water box", "n_particles": system.getNumParticles(),
                    "T_mean": diag["T_mean"], "density": diag["density"], "finite": diag["finite"],
                    "status": "PASS"})


for model, offxml in [("OPC", "opc.offxml"), ("TIP5P", "tip5p.offxml")]:
    run_water(model, offxml)


[OPC] 150 waters, 600 particles
  E_min=-1861.3  E_nvt=-1892.2 kcal/mol  finite=True
  T=275.9+/-13.6 K  density=0.988 g/cm^3



[TIP5P] 150 waters, 750 particles
  E_min=-1389.5  E_nvt=-1489.8 kcal/mol  finite=True
  T=280.4+/-11.1 K  density=1.000 g/cm^3


## D2 · One virtual-site solute in virtual-site water

A single 1,4-dichlorobenzene (carrying `BondCharge` sigma-hole sites) is solvated
in OPC water with Packmol. The ligand is parameterized by **openff-2.3.0**, which
assigns **NAGL** partial charges — the charge model users are told to use — while
OPC parameterizes the water. Virtual sites therefore appear on both species.

In [3]:
def run_one_solute():
    ff = ligand_water_ff()
    top, pos, box = pack([Molecule.from_smiles("Clc1ccc(Cl)cc1"), Molecule.from_smiles("O")],
                         [1, 150], density_kgm3=950.0)
    system = ff.create_interchange(top).to_openmm_system()
    n_vs = H.count_vsites(system)
    diag = simulate(system, pos, box, steps_nvt=1500, steps_npt=0, barostat=False)
    print(f"\n[1x dichlorobenzene (NAGL) in OPC] {system.getNumParticles()} particles, {n_vs} vsites")
    print(f"  E_min={diag['e_min']:.1f}  E_nvt={diag['e_nvt']:.1f} kcal/mol  "
          f"T={diag['T_mean']:.1f}+/-{diag['T_std']:.1f} K  finite={diag['finite']}")
    assert diag["finite"] and 150 < diag["T_mean"] < 450
    assert n_vs == 150 + 2, "expected 2 solute vsites + 1 per water"
    RESULTS.append({"notebook": "04", "system": "1x dichlorobenzene (NAGL) in OPC water",
                    "n_particles": system.getNumParticles(), "n_vsites": n_vs,
                    "T_mean": diag["T_mean"], "finite": diag["finite"], "status": "PASS"})


run_one_solute()


[1x dichlorobenzene (NAGL) in OPC] 614 particles, 152 vsites
  E_min=-1773.3  E_nvt=-1844.1 kcal/mol  T=291.2+/-11.3 K  finite=True


## D3 · Multiple virtual-site solutes in virtual-site water (Packmol)

A denser, more realistic condensed-phase system: several 1,4-dichlorobenzene
molecules (NAGL charges, `BondCharge` sites) packed together with OPC water. This
exercises many vsite-bearing solutes plus vsite water simultaneously under
dynamics.

In [4]:
def run_multi_solute(n_solute=5, n_water=200):
    ff = ligand_water_ff()
    top, pos, box = pack([Molecule.from_smiles("Clc1ccc(Cl)cc1"), Molecule.from_smiles("O")],
                         [n_solute, n_water], density_kgm3=950.0)
    system = ff.create_interchange(top).to_openmm_system()
    n_vs = H.count_vsites(system)
    diag = simulate(system, pos, box, steps_nvt=1500, steps_npt=1000)
    print(f"\n[{n_solute}x dichlorobenzene + {n_water} OPC waters] {system.getNumParticles()} particles, "
          f"{n_vs} vsites")
    print(f"  E_min={diag['e_min']:.1f}  E_nvt={diag['e_nvt']:.1f} kcal/mol  "
          f"T={diag['T_mean']:.1f}+/-{diag['T_std']:.1f} K  density={diag['density']:.3f} g/cm^3  "
          f"finite={diag['finite']}")
    assert diag["finite"], "multi-solute: non-finite dynamics"
    assert 150 < diag["T_mean"] < 450, f"multi-solute temperature {diag['T_mean']}"
    assert n_vs == n_water + 2 * n_solute, "expected vsites on every solute and water"
    assert 0.6 < diag["density"] < 1.6, f"multi-solute density {diag['density']}"
    RESULTS.append({"notebook": "04", "system": f"{n_solute}x dichlorobenzene + {n_water} OPC waters",
                    "n_particles": system.getNumParticles(), "n_vsites": n_vs,
                    "T_mean": diag["T_mean"], "density": diag["density"], "finite": diag["finite"],
                    "status": "PASS"})


run_multi_solute()


[5x dichlorobenzene + 200 OPC waters] 870 particles, 210 vsites
  E_min=-2350.7  E_nvt=-2461.6 kcal/mol  T=282.8+/-13.3 K  density=1.030 g/cm^3  finite=True


## D4 · Interchange vs `openmmforcefields` box energy (consistency)

A final cross-check that the two export pathways agree on the potential energy of
a Packmol-solvated OPC water box, reinforcing that nothing diverges once the
systems are large and periodic.

In [5]:
def openmm_water_box(xml, coords_nm, box, n):
    top = OMMTopology()
    ch = top.addChain()
    for _ in range(n):
        r = top.addResidue("HOH", ch)
        O = top.addAtom("O", Element.getBySymbol("O"), r)
        h1 = top.addAtom("H1", Element.getBySymbol("H"), r)
        h2 = top.addAtom("H2", Element.getBySymbol("H"), r)
        top.addBond(O, h1); top.addBond(O, h2)
    top.setPeriodicBoxVectors(box * u.nanometer)
    ff = OMMFF(xml)
    mod = Modeller(top, coords_nm)
    mod.addExtraParticles(ff)
    system = ff.createSystem(mod.topology, nonbondedMethod=PME, rigidWater=True, constraints=None)
    return system, mod.positions


def cross_check_box(n=120):
    off_top, pos, box = pack([Molecule.from_smiles("O")], [n], density_kgm3=1000.0)
    ic = OFF("opc.offxml").create_interchange(off_top).to_openmm_system()
    og, omm_pos = openmm_water_box("amber/opc_standard.xml", pos, box, n)
    cutoff = box[0, 0] / 2 - 0.02
    for s in (ic, og):
        for f in s.getForces():
            if isinstance(f, openmm.NonbondedForce):
                f.setUseSwitchingFunction(False)
                f.setCutoffDistance(cutoff * u.nanometer)
                f.setEwaldErrorTolerance(1e-6)
        s.setDefaultPeriodicBoxVectors(*(box * u.nanometer))
    real = H.real_atom_positions(og, omm_pos)
    e_ic, _ = H.compute_energy(ic, real, recompute_vsites=True, platform="CPU")
    e_og, _ = H.compute_energy(og, omm_pos, recompute_vsites=True, platform="CPU")
    ei = e_ic["total"].value_in_unit(H.ENERGY_UNIT)
    eo = e_og["total"].value_in_unit(H.ENERGY_UNIT)
    rel = abs(ei - eo) / max(abs(eo), 1.0)
    print(f"\n[OPC box consistency] Interchange={ei:.3f}  openmmforcefields={eo:.3f} kcal/mol  "
          f"rel diff={rel:.2e}")
    assert rel < 1e-4, f"OPC box energy mismatch {rel:.2e}"
    RESULTS.append({"notebook": "04", "system": "OPC box IC vs OMMFFS", "rel_diff": rel, "status": "PASS"})


cross_check_box()


[OPC box consistency] Interchange=2160.579  openmmforcefields=2160.579 kcal/mol  rel diff=3.59e-08


In [6]:
import json
with open("results_04.json", "w") as fh:
    json.dump(RESULTS, fh, indent=2, default=str)
print(f"\nWrote results_04.json ({len(RESULTS)} rows)")
print("Notebook 04 complete: Packmol-solvated virtual-site systems (water; one and many "
      "vsite solutes in vsite water) run stably with sane temperature/density, and the two "
      "export pathways stay consistent.")


Wrote results_04.json (5 rows)
Notebook 04 complete: Packmol-solvated virtual-site systems (water; one and many vsite solutes in vsite water) run stably with sane temperature/density, and the two export pathways stay consistent.
